In [3]:
# ==========================================
# GEE Python API: Decadal Urban Thermal Comfort Analysis
# Study Area: Dhaka City (2005, 2015, 2025)
# ==========================================

import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import xarray as xr
import rioxarray

# ---------------------------------------------------------
# STEP 1: Initialization and Study Area Setup
# ---------------------------------------------------------
# ee.Authenticate() # Uncomment this line if executing for the very first time.
ee.Initialize()

# Function to mask clouds in Landsat Collection 2 using the QA_PIXEL band
# Methodological justification: Prevents cloud anomalies from skewing the mean LST downward.
def mask_clouds(image):
    qa = image.select('QA_PIXEL')
    # Bit 3 is cloud, Bit 4 is cloud shadow
    cloud_shadow_bit = 1 << 4
    clouds_bit = 1 << 3
    # Both must be set to 0 (clear conditions)
    mask = qa.bitwiseAnd(cloud_shadow_bit).eq(0).And(qa.bitwiseAnd(clouds_bit).eq(0))
    return image.updateMask(mask)
    
# Defines the spatial extent of the study area using the FAO GAUL dataset.
dhaka = ee.FeatureCollection("FAO/GAUL/2015/level2") \
    .filter(ee.Filter.eq('ADM1_NAME', 'Dhaka')) \
    .filter(ee.Filter.eq('ADM2_NAME', 'Dhaka'))

# Generates 500 uniformly distributed random points.
# This constant feature collection ensures exact geographic coordinate sampling across all epochs.
sample_points = ee.FeatureCollection.randomPoints(dhaka.geometry(), 500)

# ---------------------------------------------------------
# STEP 2: 2005 Analysis (Landsat 5)
# ---------------------------------------------------------
print("Processing 2005 Data...")

# Defines a function to calculate LST for Landsat 5 TM.
def calc_lst_l5(image):
    st = image.select('ST_B6').multiply(0.00341802).add(149.0).subtract(273.15)
    return st.rename('LST')

l5_2005 = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2") \
    .filterBounds(dhaka) \
    .filterDate('2005-03-01', '2005-05-31') \
    .map(mask_clouds)

lst_2005 = l5_2005.map(calc_lst_l5).mean().clip(dhaka)

# Computes UTFVI
mean_lst_2005 = ee.Number(lst_2005.reduceRegion(ee.Reducer.mean(), dhaka.geometry(), 30).get('LST'))
utfvi_2005 = lst_2005.subtract(mean_lst_2005).divide(mean_lst_2005).rename('UTFVI')

# Retrieves ERA5-HEAT UTCI
era5_2005 = ee.ImageCollection("projects/climate-engine-pro/assets/ce-era5-heat") \
    .filterBounds(dhaka).filterDate('2005-03-01', '2005-05-31')
utci_2005 = era5_2005.select('utci_max').mean().subtract(273.15).clip(dhaka).rename('UTCI')

# Stack and extract pixel values
img_2005 = lst_2005.addBands(utfvi_2005).addBands(utci_2005)

# Robust extraction using getInfo()
sampled_dict_2005 = img_2005.sampleRegions(collection=sample_points, scale=30, geometries=False).getInfo()
data_2005 = pd.DataFrame([feat['properties'] for feat in sampled_dict_2005['features']]).dropna()
data_2005.to_csv('Dhaka_Data_2005.csv', index=False)

# Exports individual 2005 TIFs
geemap.ee_export_image(utfvi_2005, filename='Dhaka_UTFVI_2005.tif', scale=100, region=dhaka.geometry())
geemap.ee_export_image(utci_2005, filename='Dhaka_UTCI_2005.tif', scale=100, region=dhaka.geometry())

# ---------------------------------------------------------
# STEP 3: 2015 Analysis (Landsat 8)
# ---------------------------------------------------------
print("Processing 2015 Data...")

# Defines LST calculation for Landsat 8 TIRS.
def calc_lst_l8(image):
    st = image.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15)
    return st.rename('LST')

l8_2015 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
    .filterBounds(dhaka) \
    .filterDate('2015-03-01', '2015-05-31') \
    .map(mask_clouds)

lst_2015 = l8_2015.map(calc_lst_l8).mean().clip(dhaka)

# Computes UTFVI
mean_lst_2015 = ee.Number(lst_2015.reduceRegion(ee.Reducer.mean(), dhaka.geometry(), 30).get('LST'))
utfvi_2015 = lst_2015.subtract(mean_lst_2015).divide(mean_lst_2015).rename('UTFVI')

# Retrieves ERA5-HEAT UTCI
era5_2015 = ee.ImageCollection("projects/climate-engine-pro/assets/ce-era5-heat") \
    .filterBounds(dhaka).filterDate('2015-03-01', '2015-05-31')
utci_2015 = era5_2015.select('utci_max').mean().subtract(273.15).clip(dhaka).rename('UTCI')

# Stack and extract pixel values
img_2015 = lst_2015.addBands(utfvi_2015).addBands(utci_2015)

# Robust extraction using getInfo()
sampled_dict_2015 = img_2015.sampleRegions(collection=sample_points, scale=30, geometries=False).getInfo()
data_2015 = pd.DataFrame([feat['properties'] for feat in sampled_dict_2015['features']]).dropna()
data_2015.to_csv('Dhaka_Data_2015.csv', index=False)

# Exports individual 2015 TIFs
geemap.ee_export_image(utfvi_2015, filename='Dhaka_UTFVI_2015.tif', scale=100, region=dhaka.geometry())
geemap.ee_export_image(utci_2015, filename='Dhaka_UTCI_2015.tif', scale=100, region=dhaka.geometry())

# ---------------------------------------------------------
# STEP 4: 2025 Analysis (Landsat 8)
# ---------------------------------------------------------
print("Processing 2025 Data...")

l8_2025 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
    .filterBounds(dhaka) \
    .filterDate('2025-03-01', '2025-05-31') \
    .map(mask_clouds)

lst_2025 = l8_2025.map(calc_lst_l8).mean().clip(dhaka)

# Computes UTFVI
mean_lst_2025 = ee.Number(lst_2025.reduceRegion(ee.Reducer.mean(), dhaka.geometry(), 30).get('LST'))
utfvi_2025 = lst_2025.subtract(mean_lst_2025).divide(mean_lst_2025).rename('UTFVI')

# Retrieves ERA5-HEAT UTCI
era5_2025 = ee.ImageCollection("projects/climate-engine-pro/assets/ce-era5-heat") \
    .filterBounds(dhaka).filterDate('2025-03-01', '2025-05-31')
utci_2025 = era5_2025.select('utci_max').mean().subtract(273.15).clip(dhaka).rename('UTCI')

# Stack and extract pixel values
img_2025 = lst_2025.addBands(utfvi_2025).addBands(utci_2025)

# Robust extraction using getInfo()
sampled_dict_2025 = img_2025.sampleRegions(collection=sample_points, scale=30, geometries=False).getInfo()
data_2025 = pd.DataFrame([feat['properties'] for feat in sampled_dict_2025['features']]).dropna()
data_2025.to_csv('Dhaka_Data_2025.csv', index=False)

# Exports individual 2025 TIFs
geemap.ee_export_image(utfvi_2025, filename='Dhaka_UTFVI_2025.tif', scale=100, region=dhaka.geometry())
geemap.ee_export_image(utci_2025, filename='Dhaka_UTCI_2025.tif', scale=100, region=dhaka.geometry())

# ---------------------------------------------------------
# STEP 5 & 6: Individual Map Exports (Optional/Skipped for brevity)
# (We proceed directly to the Master Grid as it is the final publication requirement)
# ---------------------------------------------------------

# ---------------------------------------------------------
# STEP 7: Master 3x3 Grid Cartographic Export
# ---------------------------------------------------------
print("Generating Master 3x3 Publication Grid...")

# Initialize the 3x3 figure architecture. 
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(18, 18))
years = ['2005', '2015', '2025']

for i, year in enumerate(years):
    
    # ==========================================
    # ROW 1: UTFVI Maps (Top Row)
    # ==========================================
    da_utfvi = rioxarray.open_rasterio(f'Dhaka_UTFVI_{year}.tif').squeeze()
    plot_utfvi = da_utfvi.plot(ax=axes[0, i], cmap='inferno', add_colorbar=False)
    
    cbar_utfvi = fig.colorbar(plot_utfvi, ax=axes[0, i], fraction=0.046, pad=0.04)
    cbar_utfvi.set_label('UTFVI (Degradation)', rotation=270, labelpad=15, fontsize=11)
    axes[0, i].set_title(f'UTFVI ({year})', fontsize=16, fontweight='bold', pad=10)

    # ==========================================
    # ROW 2: UTCI Maps (Middle Row)
    # ==========================================
    da_utci = rioxarray.open_rasterio(f'Dhaka_UTCI_{year}.tif').squeeze()
    plot_utci = da_utci.plot(ax=axes[1, i], cmap='Spectral_r', add_colorbar=False)
    
    cbar_utci = fig.colorbar(plot_utci, ax=axes[1, i], fraction=0.046, pad=0.04)
    cbar_utci.set_label('UTCI (°C)', rotation=270, labelpad=15, fontsize=11)
    axes[1, i].set_title(f'UTCI ({year})', fontsize=16, fontweight='bold', pad=10)

    # ==========================================
    # ROW 3: Correlation Scatter Plots (Bottom Row)
    # ==========================================
    df = pd.read_csv(f'Dhaka_Data_{year}.csv')
    axes[2, i].scatter(df['UTFVI'], df['UTCI'], alpha=0.6, color='darkred', edgecolor='black', s=20)
    
    axes[2, i].set_xlabel('Urban Thermal Field Variance Index (UTFVI)', fontsize=12)
    axes[2, i].set_ylabel('Universal Thermal Climate Index (°C)', fontsize=12)
    axes[2, i].set_title(f'UTFVI vs UTCI Correlation ({year})', fontsize=14, fontweight='bold', pad=10)
    axes[2, i].grid(True, linestyle='--', alpha=0.6)
    
    # Set fixed Y and X limits to ensure the scales match perfectly across all three years
    axes[2, i].set_xlim([df['UTFVI'].min() - 0.01, df['UTFVI'].max() + 0.01])
    axes[2, i].set_ylim([df['UTCI'].min() - 2, df['UTCI'].max() + 2])

    # ==========================================
    # CARTOGRAPHIC ELEMENTS (Applied to Maps Only: Rows 0 and 1)
    # ==========================================
    for row in [0, 1]:
        axes[row, i].set_xlabel('Longitude', fontsize=10)
        axes[row, i].set_ylabel('Latitude', fontsize=10)
        axes[row, i].tick_params(axis='both', labelsize=9)
        
        for spine in axes[row, i].spines.values():
            spine.set_edgecolor('black')
            spine.set_linewidth(1.5)
        
        # Professional geometric North Arrow
        x_na, y_na = 0.08, 0.88
        axes[row, i].arrow(x_na, y_na, 0, 0.06, transform=axes[row, i].transAxes, 
                           color='black', head_width=0.03, head_length=0.04, linewidth=2, zorder=10)
        axes[row, i].text(x_na, y_na + 0.12, 'N', transform=axes[row, i].transAxes, 
                          ha='center', va='center', fontsize=14, fontweight='bold', zorder=10)

# Apply a neatline to the statistical plots
for i in range(3):
    for spine in axes[2, i].spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1.5)

plt.tight_layout(pad=3.0)
plt.savefig('Master_Thermal_Grid_Dhaka_2005_2025.jpg', dpi=300, bbox_inches='tight')
plt.close()

print("Master 3x3 Grid successfully saved as 'Master_Thermal_Grid_Dhaka_2005_2025.jpg'.")

Processing 2005 Data...
Generating URL ...
Please wait ...
Data downloaded to I:\Dhaka_UTFVI_2005.tif
Generating URL ...
Please wait ...
Data downloaded to I:\Dhaka_UTCI_2005.tif
Processing 2015 Data...
Generating URL ...
Please wait ...
Data downloaded to I:\Dhaka_UTFVI_2015.tif
Generating URL ...
Please wait ...
Data downloaded to I:\Dhaka_UTCI_2015.tif
Processing 2025 Data...
Generating URL ...
Please wait ...
Data downloaded to I:\Dhaka_UTFVI_2025.tif
Generating URL ...
Please wait ...
Data downloaded to I:\Dhaka_UTCI_2025.tif
Generating Master 3x3 Publication Grid...
Master 3x3 Grid successfully saved as 'Master_Thermal_Grid_Dhaka_2005_2025.jpg'.


In [5]:
# ==========================================
# GEE Python API: Decadal Urban Thermal Comfort Analysis
# Study Area: Dhaka City (2005, 2015, 2025)
# Output: Individual Maps, Graphs, and Tables
# ==========================================

import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import xarray as xr
import rioxarray

# ---------------------------------------------------------
# STEP 1: Initialization and Study Area Setup
# ---------------------------------------------------------
# ee.Authenticate() # Uncomment if executing for the very first time.
ee.Initialize()

# Function to mask clouds in Landsat Collection 2 using the QA_PIXEL band
def mask_clouds(image):
    qa = image.select('QA_PIXEL')
    cloud_shadow_bit = 1 << 4
    clouds_bit = 1 << 3
    mask = qa.bitwiseAnd(cloud_shadow_bit).eq(0).And(qa.bitwiseAnd(clouds_bit).eq(0))
    return image.updateMask(mask)
    
# Defines the spatial extent of the study area
dhaka = ee.FeatureCollection("FAO/GAUL/2015/level2") \
    .filter(ee.Filter.eq('ADM1_NAME', 'Dhaka')) \
    .filter(ee.Filter.eq('ADM2_NAME', 'Dhaka'))

# Generates 500 uniformly distributed random points for consistent extraction
sample_points = ee.FeatureCollection.randomPoints(dhaka.geometry(), 500)

# ---------------------------------------------------------
# STEP 2: 2005 Processing & Individual Data/Graph Exports
# ---------------------------------------------------------
print("Processing 2005 Data...")

def calc_lst_l5(image):
    st = image.select('ST_B6').multiply(0.00341802).add(149.0).subtract(273.15)
    return st.rename('LST')

l5_2005 = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2") \
    .filterBounds(dhaka).filterDate('2005-03-01', '2005-05-31').map(mask_clouds)
lst_2005 = l5_2005.map(calc_lst_l5).mean().clip(dhaka)

mean_lst_2005 = ee.Number(lst_2005.reduceRegion(ee.Reducer.mean(), dhaka.geometry(), 30).get('LST'))
utfvi_2005 = lst_2005.subtract(mean_lst_2005).divide(mean_lst_2005).rename('UTFVI')

era5_2005 = ee.ImageCollection("projects/climate-engine-pro/assets/ce-era5-heat") \
    .filterBounds(dhaka).filterDate('2005-03-01', '2005-05-31')
utci_2005 = era5_2005.select('utci_max').mean().subtract(273.15).clip(dhaka).rename('UTCI')

img_2005 = lst_2005.addBands(utfvi_2005).addBands(utci_2005)

# EXPORT 1: Individual CSV Table (2005)
sampled_dict_2005 = img_2005.sampleRegions(collection=sample_points, scale=30, geometries=False).getInfo()
data_2005 = pd.DataFrame([feat['properties'] for feat in sampled_dict_2005['features']]).dropna()
data_2005.to_csv('Table_Dhaka_Data_2005.csv', index=False)
print("Exported: Table_Dhaka_Data_2005.csv")

# EXPORT 2: Individual Correlation Graph (2005)
plt.figure(figsize=(8, 6))
plt.scatter(data_2005['UTFVI'], data_2005['UTCI'], alpha=0.6, color='blue', edgecolor='black')
plt.xlabel('Urban Thermal Field Variance Index (UTFVI)', fontsize=12)
plt.ylabel('Universal Thermal Climate Index (°C)', fontsize=12)
plt.title('2005: Surface Degradation vs Human Heat Stress', fontsize=14, fontweight='bold')
plt.grid(True, linestyle='--')
plt.tight_layout()
plt.savefig('Graph_Correlation_2005.jpg', dpi=300)
plt.close()
print("Exported: Graph_Correlation_2005.jpg")

# Download Raw TIFs for mapping
geemap.ee_export_image(utfvi_2005, filename='Dhaka_UTFVI_2005.tif', scale=100, region=dhaka.geometry())
geemap.ee_export_image(utci_2005, filename='Dhaka_UTCI_2005.tif', scale=100, region=dhaka.geometry())


# ---------------------------------------------------------
# STEP 3: 2015 Processing & Individual Data/Graph Exports
# ---------------------------------------------------------
print("Processing 2015 Data...")

def calc_lst_l8(image):
    st = image.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15)
    return st.rename('LST')

l8_2015 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
    .filterBounds(dhaka).filterDate('2015-03-01', '2015-05-31').map(mask_clouds)
lst_2015 = l8_2015.map(calc_lst_l8).mean().clip(dhaka)

mean_lst_2015 = ee.Number(lst_2015.reduceRegion(ee.Reducer.mean(), dhaka.geometry(), 30).get('LST'))
utfvi_2015 = lst_2015.subtract(mean_lst_2015).divide(mean_lst_2015).rename('UTFVI')

era5_2015 = ee.ImageCollection("projects/climate-engine-pro/assets/ce-era5-heat") \
    .filterBounds(dhaka).filterDate('2015-03-01', '2015-05-31')
utci_2015 = era5_2015.select('utci_max').mean().subtract(273.15).clip(dhaka).rename('UTCI')

img_2015 = lst_2015.addBands(utfvi_2015).addBands(utci_2015)

# EXPORT 3: Individual CSV Table (2015)
sampled_dict_2015 = img_2015.sampleRegions(collection=sample_points, scale=30, geometries=False).getInfo()
data_2015 = pd.DataFrame([feat['properties'] for feat in sampled_dict_2015['features']]).dropna()
data_2015.to_csv('Table_Dhaka_Data_2015.csv', index=False)
print("Exported: Table_Dhaka_Data_2015.csv")

# EXPORT 4: Individual Correlation Graph (2015)
plt.figure(figsize=(8, 6))
plt.scatter(data_2015['UTFVI'], data_2015['UTCI'], alpha=0.6, color='orange', edgecolor='black')
plt.xlabel('Urban Thermal Field Variance Index (UTFVI)', fontsize=12)
plt.ylabel('Universal Thermal Climate Index (°C)', fontsize=12)
plt.title('2015: Surface Degradation vs Human Heat Stress', fontsize=14, fontweight='bold')
plt.grid(True, linestyle='--')
plt.tight_layout()
plt.savefig('Graph_Correlation_2015.jpg', dpi=300)
plt.close()
print("Exported: Graph_Correlation_2015.jpg")

# Download Raw TIFs for mapping
geemap.ee_export_image(utfvi_2015, filename='Dhaka_UTFVI_2015.tif', scale=100, region=dhaka.geometry())
geemap.ee_export_image(utci_2015, filename='Dhaka_UTCI_2015.tif', scale=100, region=dhaka.geometry())


# ---------------------------------------------------------
# STEP 4: 2025 Processing & Individual Data/Graph Exports
# ---------------------------------------------------------
print("Processing 2025 Data...")

l8_2025 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
    .filterBounds(dhaka).filterDate('2025-03-01', '2025-05-31').map(mask_clouds)
lst_2025 = l8_2025.map(calc_lst_l8).mean().clip(dhaka)

mean_lst_2025 = ee.Number(lst_2025.reduceRegion(ee.Reducer.mean(), dhaka.geometry(), 30).get('LST'))
utfvi_2025 = lst_2025.subtract(mean_lst_2025).divide(mean_lst_2025).rename('UTFVI')

era5_2025 = ee.ImageCollection("projects/climate-engine-pro/assets/ce-era5-heat") \
    .filterBounds(dhaka).filterDate('2025-03-01', '2025-05-31')
utci_2025 = era5_2025.select('utci_max').mean().subtract(273.15).clip(dhaka).rename('UTCI')

img_2025 = lst_2025.addBands(utfvi_2025).addBands(utci_2025)

# EXPORT 5: Individual CSV Table (2025)
sampled_dict_2025 = img_2025.sampleRegions(collection=sample_points, scale=30, geometries=False).getInfo()
data_2025 = pd.DataFrame([feat['properties'] for feat in sampled_dict_2025['features']]).dropna()
data_2025.to_csv('Table_Dhaka_Data_2025.csv', index=False)
print("Exported: Table_Dhaka_Data_2025.csv")

# EXPORT 6: Individual Correlation Graph (2025)
plt.figure(figsize=(8, 6))
plt.scatter(data_2025['UTFVI'], data_2025['UTCI'], alpha=0.6, color='darkred', edgecolor='black')
plt.xlabel('Urban Thermal Field Variance Index (UTFVI)', fontsize=12)
plt.ylabel('Universal Thermal Climate Index (°C)', fontsize=12)
plt.title('2025: Surface Degradation vs Human Heat Stress', fontsize=14, fontweight='bold')
plt.grid(True, linestyle='--')
plt.tight_layout()
plt.savefig('Graph_Correlation_2025.jpg', dpi=300)
plt.close()
print("Exported: Graph_Correlation_2025.jpg")

# Download Raw TIFs for mapping
geemap.ee_export_image(utfvi_2025, filename='Dhaka_UTFVI_2025.tif', scale=100, region=dhaka.geometry())
geemap.ee_export_image(utci_2025, filename='Dhaka_UTCI_2025.tif', scale=100, region=dhaka.geometry())


# ---------------------------------------------------------
# STEP 5: Exporting Individual Publication-Ready UTFVI Maps
# ---------------------------------------------------------
print("Generating Individual UTFVI Maps...")
years = ['2005', '2015', '2025']

for year in years:
    da_utfvi = rioxarray.open_rasterio(f'Dhaka_UTFVI_{year}.tif').squeeze()
    
    fig, ax = plt.subplots(figsize=(8, 8))
    plot_utfvi = da_utfvi.plot(ax=ax, cmap='inferno', add_colorbar=False)
    
    # Cartographic Elements
    cbar = fig.colorbar(plot_utfvi, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Urban Thermal Field Variance Index (UTFVI)', rotation=270, labelpad=15, fontsize=12)
    ax.set_title(f'Spatial Distribution of UTFVI\nDhaka City ({year})', fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel('Longitude', fontsize=12)
    ax.set_ylabel('Latitude', fontsize=12)
    
    # Neatline
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(2)
        
    # North Arrow
    ax.arrow(0.08, 0.88, 0, 0.05, transform=ax.transAxes, color='black', head_width=0.03, head_length=0.04, linewidth=2)
    ax.text(0.08, 0.96, 'N', transform=ax.transAxes, ha='center', va='center', fontsize=16, fontweight='bold')
    
    plt.tight_layout()
    map_filename = f'Map_Standalone_UTFVI_{year}.jpg'
    plt.savefig(map_filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Exported: {map_filename}")

# ---------------------------------------------------------
# STEP 6: Exporting Individual Publication-Ready UTCI Maps
# ---------------------------------------------------------
print("Generating Individual UTCI Maps...")

for year in years:
    da_utci = rioxarray.open_rasterio(f'Dhaka_UTCI_{year}.tif').squeeze()
    
    fig, ax = plt.subplots(figsize=(8, 8))
    plot_utci = da_utci.plot(ax=ax, cmap='Spectral_r', add_colorbar=False)
    
    # Cartographic Elements
    cbar = fig.colorbar(plot_utci, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Universal Thermal Climate Index (°C)', rotation=270, labelpad=15, fontsize=12)
    ax.set_title(f'Spatial Distribution of Human Heat Stress (UTCI)\nDhaka City ({year})', fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel('Longitude', fontsize=12)
    ax.set_ylabel('Latitude', fontsize=12)
    
    # Neatline
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(2)
        
    # North Arrow
    ax.arrow(0.08, 0.88, 0, 0.05, transform=ax.transAxes, color='black', head_width=0.03, head_length=0.04, linewidth=2)
    ax.text(0.08, 0.96, 'N', transform=ax.transAxes, ha='center', va='center', fontsize=16, fontweight='bold')
    
    plt.tight_layout()
    map_filename = f'Map_Standalone_UTCI_{year}.jpg'
    plt.savefig(map_filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Exported: {map_filename}")

print("All standalone files (Maps, Graphs, Tables) successfully exported.")

Processing 2005 Data...
Exported: Table_Dhaka_Data_2005.csv
Exported: Graph_Correlation_2005.jpg
Generating URL ...
Please wait ...
Data downloaded to I:\Dhaka_UTFVI_2005.tif
Generating URL ...
Please wait ...
Data downloaded to I:\Dhaka_UTCI_2005.tif
Processing 2015 Data...
Exported: Table_Dhaka_Data_2015.csv
Exported: Graph_Correlation_2015.jpg
Generating URL ...
Please wait ...
Data downloaded to I:\Dhaka_UTFVI_2015.tif
Generating URL ...
Please wait ...
Data downloaded to I:\Dhaka_UTCI_2015.tif
Processing 2025 Data...
Exported: Table_Dhaka_Data_2025.csv
Exported: Graph_Correlation_2025.jpg
Generating URL ...
Please wait ...
Data downloaded to I:\Dhaka_UTFVI_2025.tif
Generating URL ...
Please wait ...
Data downloaded to I:\Dhaka_UTCI_2025.tif
Generating Individual UTFVI Maps...
Exported: Map_Standalone_UTFVI_2005.jpg
Exported: Map_Standalone_UTFVI_2015.jpg
Exported: Map_Standalone_UTFVI_2025.jpg
Generating Individual UTCI Maps...
Exported: Map_Standalone_UTCI_2005.jpg
Exported: Map_

In [6]:
# ==========================================
# GEE Python API: Decadal Urban Thermal Comfort Analysis
# Output: Individual Maps (White BG, Stretched), Trend Line Graphs, Tables
# ==========================================

import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import xarray as xr
import rioxarray

# ---------------------------------------------------------
# STEP 1: Initialization and Study Area Setup
# ---------------------------------------------------------
# ee.Authenticate() # Uncomment if executing for the very first time.
ee.Initialize()

def mask_clouds(image):
    qa = image.select('QA_PIXEL')
    cloud_shadow_bit = 1 << 4
    clouds_bit = 1 << 3
    mask = qa.bitwiseAnd(cloud_shadow_bit).eq(0).And(qa.bitwiseAnd(clouds_bit).eq(0))
    return image.updateMask(mask)
    
dhaka = ee.FeatureCollection("FAO/GAUL/2015/level2") \
    .filter(ee.Filter.eq('ADM1_NAME', 'Dhaka')) \
    .filter(ee.Filter.eq('ADM2_NAME', 'Dhaka'))

sample_points = ee.FeatureCollection.randomPoints(dhaka.geometry(), 500)

# ---------------------------------------------------------
# STEP 2: 2005 Processing & Exports
# ---------------------------------------------------------
print("Processing 2005 Data...")

def calc_lst_l5(image):
    st = image.select('ST_B6').multiply(0.00341802).add(149.0).subtract(273.15)
    return st.rename('LST')

l5_2005 = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2") \
    .filterBounds(dhaka).filterDate('2005-03-01', '2005-05-31').map(mask_clouds)
lst_2005 = l5_2005.map(calc_lst_l5).mean().clip(dhaka)

mean_lst_2005 = ee.Number(lst_2005.reduceRegion(ee.Reducer.mean(), dhaka.geometry(), 30).get('LST'))
utfvi_2005 = lst_2005.subtract(mean_lst_2005).divide(mean_lst_2005).rename('UTFVI')

era5_2005 = ee.ImageCollection("projects/climate-engine-pro/assets/ce-era5-heat") \
    .filterBounds(dhaka).filterDate('2005-03-01', '2005-05-31')
utci_2005 = era5_2005.select('utci_max').mean().subtract(273.15).clip(dhaka).rename('UTCI')

img_2005 = lst_2005.addBands(utfvi_2005).addBands(utci_2005)

# Extract Data
sampled_dict_2005 = img_2005.sampleRegions(collection=sample_points, scale=30, geometries=False).getInfo()
data_2005 = pd.DataFrame([feat['properties'] for feat in sampled_dict_2005['features']]).dropna()
data_2005.to_csv('Table_Dhaka_Data_2005.csv', index=False)

# Export Map TIFs with explicit NoData values (-9999) to clear the background
geemap.ee_export_image(utfvi_2005.unmask(-9999), filename='Dhaka_UTFVI_2005.tif', scale=100, region=dhaka.geometry())
geemap.ee_export_image(utci_2005.unmask(-9999), filename='Dhaka_UTCI_2005.tif', scale=100, region=dhaka.geometry())


# ---------------------------------------------------------
# STEP 3: 2015 Processing & Exports
# ---------------------------------------------------------
print("Processing 2015 Data...")

def calc_lst_l8(image):
    st = image.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15)
    return st.rename('LST')

l8_2015 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
    .filterBounds(dhaka).filterDate('2015-03-01', '2015-05-31').map(mask_clouds)
lst_2015 = l8_2015.map(calc_lst_l8).mean().clip(dhaka)

mean_lst_2015 = ee.Number(lst_2015.reduceRegion(ee.Reducer.mean(), dhaka.geometry(), 30).get('LST'))
utfvi_2015 = lst_2015.subtract(mean_lst_2015).divide(mean_lst_2015).rename('UTFVI')

era5_2015 = ee.ImageCollection("projects/climate-engine-pro/assets/ce-era5-heat") \
    .filterBounds(dhaka).filterDate('2015-03-01', '2015-05-31')
utci_2015 = era5_2015.select('utci_max').mean().subtract(273.15).clip(dhaka).rename('UTCI')

img_2015 = lst_2015.addBands(utfvi_2015).addBands(utci_2015)

# Extract Data
sampled_dict_2015 = img_2015.sampleRegions(collection=sample_points, scale=30, geometries=False).getInfo()
data_2015 = pd.DataFrame([feat['properties'] for feat in sampled_dict_2015['features']]).dropna()
data_2015.to_csv('Table_Dhaka_Data_2015.csv', index=False)

# Export Map TIFs
geemap.ee_export_image(utfvi_2015.unmask(-9999), filename='Dhaka_UTFVI_2015.tif', scale=100, region=dhaka.geometry())
geemap.ee_export_image(utci_2015.unmask(-9999), filename='Dhaka_UTCI_2015.tif', scale=100, region=dhaka.geometry())


# ---------------------------------------------------------
# STEP 4: 2025 Processing & Exports
# ---------------------------------------------------------
print("Processing 2025 Data...")

l8_2025 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
    .filterBounds(dhaka).filterDate('2025-03-01', '2025-05-31').map(mask_clouds)
lst_2025 = l8_2025.map(calc_lst_l8).mean().clip(dhaka)

mean_lst_2025 = ee.Number(lst_2025.reduceRegion(ee.Reducer.mean(), dhaka.geometry(), 30).get('LST'))
utfvi_2025 = lst_2025.subtract(mean_lst_2025).divide(mean_lst_2025).rename('UTFVI')

era5_2025 = ee.ImageCollection("projects/climate-engine-pro/assets/ce-era5-heat") \
    .filterBounds(dhaka).filterDate('2025-03-01', '2025-05-31')
utci_2025 = era5_2025.select('utci_max').mean().subtract(273.15).clip(dhaka).rename('UTCI')

img_2025 = lst_2025.addBands(utfvi_2025).addBands(utci_2025)

# Extract Data
sampled_dict_2025 = img_2025.sampleRegions(collection=sample_points, scale=30, geometries=False).getInfo()
data_2025 = pd.DataFrame([feat['properties'] for feat in sampled_dict_2025['features']]).dropna()
data_2025.to_csv('Table_Dhaka_Data_2025.csv', index=False)

# Export Map TIFs
geemap.ee_export_image(utfvi_2025.unmask(-9999), filename='Dhaka_UTFVI_2025.tif', scale=100, region=dhaka.geometry())
geemap.ee_export_image(utci_2025.unmask(-9999), filename='Dhaka_UTCI_2025.tif', scale=100, region=dhaka.geometry())


# ---------------------------------------------------------
# STEP 5: Generating Binned Trend Line Graphs
# ---------------------------------------------------------
print("Generating Binned Trend Line Graphs...")
datasets = {'2005': (data_2005, 'blue'), '2015': (data_2015, 'orange'), '2025': (data_2025, 'darkred')}

for year, (df, color) in datasets.items():
    # Sort and bin the data to create a smooth trend line
    df_sorted = df.sort_values(by='UTFVI').copy()
    df_sorted['UTFVI_bins'] = pd.cut(df_sorted['UTFVI'], bins=15)
    trend = df_sorted.groupby('UTFVI_bins', observed=True)['UTCI'].mean().reset_index()
    trend['UTFVI_mid'] = trend['UTFVI_bins'].apply(lambda x: x.mid).astype(float)
    trend = trend.dropna()

    plt.figure(figsize=(8, 6))
    plt.plot(trend['UTFVI_mid'], trend['UTCI'], color=color, linewidth=2.5, marker='o', 
             markersize=8, markerfacecolor='white', markeredgewidth=2)
    
    plt.xlabel('Urban Thermal Field Variance Index (UTFVI)', fontsize=12)
    plt.ylabel('Mean Universal Thermal Climate Index (°C)', fontsize=12)
    plt.title(f'{year}: Surface Degradation vs Human Heat Stress Trend', fontsize=14, fontweight='bold')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(f'Graph_LineTrend_{year}.jpg', dpi=300)
    plt.close()
    print(f"Exported: Graph_LineTrend_{year}.jpg")

# ---------------------------------------------------------
# STEP 6: Exporting Individual Publication-Ready Maps
# ---------------------------------------------------------
print("Generating Corrected Maps...")
years = ['2005', '2015', '2025']

for year in years:
    # --- UTFVI MAPPING ---
    da_utfvi = rioxarray.open_rasterio(f'Dhaka_UTFVI_{year}.tif').squeeze()
    da_utfvi = da_utfvi.where(da_utfvi > -9000) # Mask the NoData background
    
    fig, ax = plt.subplots(figsize=(8, 8))
    # Setting dynamic limits for UTFVI to ignore extreme outliers
    v_min = da_utfvi.quantile(0.02).values
    v_max = da_utfvi.quantile(0.98).values
    plot_utfvi = da_utfvi.plot(ax=ax, cmap='inferno', vmin=v_min, vmax=v_max, add_colorbar=False)
    
    cbar = fig.colorbar(plot_utfvi, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Urban Thermal Field Variance Index (UTFVI)', rotation=270, labelpad=15, fontsize=12)
    ax.set_title(f'Spatial Distribution of UTFVI\nDhaka City ({year})', fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel('Longitude', fontsize=12); ax.set_ylabel('Latitude', fontsize=12)
    ax.set_facecolor('white') # Ensure background is white
    
    for spine in ax.spines.values():
        spine.set_edgecolor('black'); spine.set_linewidth(2)
        
    ax.arrow(0.08, 0.88, 0, 0.05, transform=ax.transAxes, color='black', head_width=0.03, head_length=0.04, linewidth=2)
    ax.text(0.08, 0.96, 'N', transform=ax.transAxes, ha='center', va='center', fontsize=16, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(f'Map_Standalone_UTFVI_{year}.jpg', dpi=300, bbox_inches='tight')
    plt.close()

    # --- UTCI MAPPING ---
    da_utci = rioxarray.open_rasterio(f'Dhaka_UTCI_{year}.tif').squeeze()
    da_utci = da_utci.where(da_utci > -9000) # Mask the NoData background
    
    fig, ax = plt.subplots(figsize=(8, 8))
    # Dynamic Histogram Stretching to show micro-variations
    u_min = da_utci.quantile(0.02).values
    u_max = da_utci.quantile(0.98).values
    plot_utci = da_utci.plot(ax=ax, cmap='Spectral_r', vmin=u_min, vmax=u_max, add_colorbar=False)
    
    cbar = fig.colorbar(plot_utci, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Universal Thermal Climate Index (°C)', rotation=270, labelpad=15, fontsize=12)
    ax.set_title(f'Spatial Distribution of Human Heat Stress (UTCI)\nDhaka City ({year})', fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel('Longitude', fontsize=12); ax.set_ylabel('Latitude', fontsize=12)
    ax.set_facecolor('white') # Ensure background is white
    
    for spine in ax.spines.values():
        spine.set_edgecolor('black'); spine.set_linewidth(2)
        
    ax.arrow(0.08, 0.88, 0, 0.05, transform=ax.transAxes, color='black', head_width=0.03, head_length=0.04, linewidth=2)
    ax.text(0.08, 0.96, 'N', transform=ax.transAxes, ha='center', va='center', fontsize=16, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(f'Map_Standalone_UTCI_{year}.jpg', dpi=300, bbox_inches='tight')
    plt.close()

print("All corrected standalone files successfully exported.")

Processing 2005 Data...
Generating URL ...
Please wait ...
Data downloaded to I:\Dhaka_UTFVI_2005.tif
Generating URL ...
Please wait ...
Data downloaded to I:\Dhaka_UTCI_2005.tif
Processing 2015 Data...
Generating URL ...
Please wait ...
Data downloaded to I:\Dhaka_UTFVI_2015.tif
Generating URL ...
Please wait ...
Data downloaded to I:\Dhaka_UTCI_2015.tif
Processing 2025 Data...
Generating URL ...
Please wait ...
Data downloaded to I:\Dhaka_UTFVI_2025.tif
Generating URL ...
Please wait ...
Data downloaded to I:\Dhaka_UTCI_2025.tif
Generating Binned Trend Line Graphs...
Exported: Graph_LineTrend_2005.jpg
Exported: Graph_LineTrend_2015.jpg
Exported: Graph_LineTrend_2025.jpg
Generating Corrected Maps...
All corrected standalone files successfully exported.


In [7]:
# ==========================================
# GEE Python API: Decadal Urban Thermal Comfort Analysis
# Output: LST, UTFVI, UTCI (High Res), Trend Graphs, Tables
# ==========================================

import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import xarray as xr
import rioxarray

# ---------------------------------------------------------
# STEP 1: Initialization and Study Area Setup
# ---------------------------------------------------------
# ee.Authenticate() # Uncomment if executing for the very first time.
ee.Initialize()

def mask_clouds(image):
    qa = image.select('QA_PIXEL')
    cloud_shadow_bit = 1 << 4
    clouds_bit = 1 << 3
    mask = qa.bitwiseAnd(cloud_shadow_bit).eq(0).And(qa.bitwiseAnd(clouds_bit).eq(0))
    return image.updateMask(mask)
    
dhaka = ee.FeatureCollection("FAO/GAUL/2015/level2") \
    .filter(ee.Filter.eq('ADM1_NAME', 'Dhaka')) \
    .filter(ee.Filter.eq('ADM2_NAME', 'Dhaka'))

sample_points = ee.FeatureCollection.randomPoints(dhaka.geometry(), 500)

# ---------------------------------------------------------
# STEP 2: 2005 Processing & Exports
# ---------------------------------------------------------
print("Processing 2005 Data...")

def calc_lst_l5(image):
    st = image.select('ST_B6').multiply(0.00341802).add(149.0).subtract(273.15)
    return st.rename('LST')

l5_2005 = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2") \
    .filterBounds(dhaka).filterDate('2005-03-01', '2005-05-31').map(mask_clouds)
lst_2005 = l5_2005.map(calc_lst_l5).mean().clip(dhaka)

mean_lst_2005 = ee.Number(lst_2005.reduceRegion(ee.Reducer.mean(), dhaka.geometry(), 30).get('LST'))
utfvi_2005 = lst_2005.subtract(mean_lst_2005).divide(mean_lst_2005).rename('UTFVI')

# Applied bilinear resampling to smooth the 27km ERA5 data into a continuous high-res gradient
era5_2005 = ee.ImageCollection("projects/climate-engine-pro/assets/ce-era5-heat") \
    .filterBounds(dhaka).filterDate('2005-03-01', '2005-05-31')
utci_2005 = era5_2005.select('utci_max').mean().subtract(273.15).resample('bilinear').clip(dhaka).rename('UTCI')

img_2005 = lst_2005.addBands(utfvi_2005).addBands(utci_2005)

# Extract Data
sampled_dict_2005 = img_2005.sampleRegions(collection=sample_points, scale=30, geometries=False).getInfo()
data_2005 = pd.DataFrame([feat['properties'] for feat in sampled_dict_2005['features']]).dropna()
data_2005.to_csv('Table_Dhaka_Data_2005.csv', index=False)

# Export Map TIFs (Scale set to 30m for maximum resolution)
geemap.ee_export_image(lst_2005.unmask(-9999), filename='Dhaka_LST_2005.tif', scale=30, region=dhaka.geometry())
geemap.ee_export_image(utfvi_2005.unmask(-9999), filename='Dhaka_UTFVI_2005.tif', scale=30, region=dhaka.geometry())
geemap.ee_export_image(utci_2005.unmask(-9999), filename='Dhaka_UTCI_2005.tif', scale=30, region=dhaka.geometry())


# ---------------------------------------------------------
# STEP 3: 2015 Processing & Exports
# ---------------------------------------------------------
print("Processing 2015 Data...")

def calc_lst_l8(image):
    st = image.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15)
    return st.rename('LST')

l8_2015 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
    .filterBounds(dhaka).filterDate('2015-03-01', '2015-05-31').map(mask_clouds)
lst_2015 = l8_2015.map(calc_lst_l8).mean().clip(dhaka)

mean_lst_2015 = ee.Number(lst_2015.reduceRegion(ee.Reducer.mean(), dhaka.geometry(), 30).get('LST'))
utfvi_2015 = lst_2015.subtract(mean_lst_2015).divide(mean_lst_2015).rename('UTFVI')

era5_2015 = ee.ImageCollection("projects/climate-engine-pro/assets/ce-era5-heat") \
    .filterBounds(dhaka).filterDate('2015-03-01', '2015-05-31')
utci_2015 = era5_2015.select('utci_max').mean().subtract(273.15).resample('bilinear').clip(dhaka).rename('UTCI')

img_2015 = lst_2015.addBands(utfvi_2015).addBands(utci_2015)

# Extract Data
sampled_dict_2015 = img_2015.sampleRegions(collection=sample_points, scale=30, geometries=False).getInfo()
data_2015 = pd.DataFrame([feat['properties'] for feat in sampled_dict_2015['features']]).dropna()
data_2015.to_csv('Table_Dhaka_Data_2015.csv', index=False)

# Export Map TIFs
geemap.ee_export_image(lst_2015.unmask(-9999), filename='Dhaka_LST_2015.tif', scale=30, region=dhaka.geometry())
geemap.ee_export_image(utfvi_2015.unmask(-9999), filename='Dhaka_UTFVI_2015.tif', scale=30, region=dhaka.geometry())
geemap.ee_export_image(utci_2015.unmask(-9999), filename='Dhaka_UTCI_2015.tif', scale=30, region=dhaka.geometry())


# ---------------------------------------------------------
# STEP 4: 2025 Processing & Exports
# ---------------------------------------------------------
print("Processing 2025 Data...")

l8_2025 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
    .filterBounds(dhaka).filterDate('2025-03-01', '2025-05-31').map(mask_clouds)
lst_2025 = l8_2025.map(calc_lst_l8).mean().clip(dhaka)

mean_lst_2025 = ee.Number(lst_2025.reduceRegion(ee.Reducer.mean(), dhaka.geometry(), 30).get('LST'))
utfvi_2025 = lst_2025.subtract(mean_lst_2025).divide(mean_lst_2025).rename('UTFVI')

era5_2025 = ee.ImageCollection("projects/climate-engine-pro/assets/ce-era5-heat") \
    .filterBounds(dhaka).filterDate('2025-03-01', '2025-05-31')
utci_2025 = era5_2025.select('utci_max').mean().subtract(273.15).resample('bilinear').clip(dhaka).rename('UTCI')

img_2025 = lst_2025.addBands(utfvi_2025).addBands(utci_2025)

# Extract Data
sampled_dict_2025 = img_2025.sampleRegions(collection=sample_points, scale=30, geometries=False).getInfo()
data_2025 = pd.DataFrame([feat['properties'] for feat in sampled_dict_2025['features']]).dropna()
data_2025.to_csv('Table_Dhaka_Data_2025.csv', index=False)

# Export Map TIFs
geemap.ee_export_image(lst_2025.unmask(-9999), filename='Dhaka_LST_2025.tif', scale=30, region=dhaka.geometry())
geemap.ee_export_image(utfvi_2025.unmask(-9999), filename='Dhaka_UTFVI_2025.tif', scale=30, region=dhaka.geometry())
geemap.ee_export_image(utci_2025.unmask(-9999), filename='Dhaka_UTCI_2025.tif', scale=30, region=dhaka.geometry())


# ---------------------------------------------------------
# STEP 5: Generating Binned Trend Line Graphs
# ---------------------------------------------------------
print("Generating Binned Trend Line Graphs...")
datasets = {'2005': (data_2005, 'blue'), '2015': (data_2015, 'orange'), '2025': (data_2025, 'darkred')}

for year, (df, color) in datasets.items():
    df_sorted = df.sort_values(by='UTFVI').copy()
    df_sorted['UTFVI_bins'] = pd.cut(df_sorted['UTFVI'], bins=15)
    trend = df_sorted.groupby('UTFVI_bins', observed=True)['UTCI'].mean().reset_index()
    trend['UTFVI_mid'] = trend['UTFVI_bins'].apply(lambda x: x.mid).astype(float)
    trend = trend.dropna()

    plt.figure(figsize=(8, 6))
    plt.plot(trend['UTFVI_mid'], trend['UTCI'], color=color, linewidth=2.5, marker='o', 
             markersize=8, markerfacecolor='white', markeredgewidth=2)
    
    plt.xlabel('Urban Thermal Field Variance Index (UTFVI)', fontsize=12)
    plt.ylabel('Mean Universal Thermal Climate Index (°C)', fontsize=12)
    plt.title(f'{year}: Surface Degradation vs Human Heat Stress Trend', fontsize=14, fontweight='bold')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(f'Graph_LineTrend_{year}.jpg', dpi=300)
    plt.close()


# ---------------------------------------------------------
# STEP 6: Cartographic Rendering Function
# ---------------------------------------------------------
print("Generating High-Resolution Maps...")
years = ['2005', '2015', '2025']

# Reusable cartographic function to ensure perfect consistency
def render_map(filename, cmap, title, cbar_label, output_name):
    da = rioxarray.open_rasterio(filename).squeeze()
    da = da.where(da > -9000) # Mask NoData
    
    fig, ax = plt.subplots(figsize=(8, 8))
    v_min, v_max = da.quantile(0.02).values, da.quantile(0.98).values
    
    plot = da.plot(ax=ax, cmap=cmap, vmin=v_min, vmax=v_max, add_colorbar=False)
    
    cbar = fig.colorbar(plot, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(cbar_label, rotation=270, labelpad=15, fontsize=12)
    
    ax.set_title(title, fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel('Longitude', fontsize=12)
    ax.set_ylabel('Latitude', fontsize=12)
    ax.set_facecolor('white') 
    
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(2)
        
    # Corrected, Highly Visible North Arrow
    ax.annotate('N', xy=(0.08, 0.95), xytext=(0.08, 0.88),
                arrowprops=dict(facecolor='black', width=4, headwidth=12),
                ha='center', va='center', fontsize=16, fontweight='bold',
                xycoords=ax.transAxes, zorder=10, 
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="none", alpha=0.7))
    
    plt.tight_layout()
    plt.savefig(output_name, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Exported: {output_name}")

# Generate all maps using the standardized function
for year in years:
    # 1. LST Maps
    render_map(f'Dhaka_LST_{year}.tif', 'RdYlBu_r', 
               f'Spatial Distribution of LST\nDhaka City ({year})', 
               'Land Surface Temperature (°C)', f'Map_Standalone_LST_{year}.jpg')
               
    # 2. UTFVI Maps
    render_map(f'Dhaka_UTFVI_{year}.tif', 'inferno', 
               f'Spatial Distribution of UTFVI\nDhaka City ({year})', 
               'Urban Thermal Field Variance Index (UTFVI)', f'Map_Standalone_UTFVI_{year}.jpg')
               
    # 3. UTCI Maps
    render_map(f'Dhaka_UTCI_{year}.tif', 'Spectral_r', 
               f'Spatial Distribution of Human Heat Stress (UTCI)\nDhaka City ({year})', 
               'Universal Thermal Climate Index (°C)', f'Map_Standalone_UTCI_{year}.jpg')

print("All advanced standalone files successfully exported.")

Processing 2005 Data...
Generating URL ...
Please wait ...
Data downloaded to I:\Dhaka_LST_2005.tif
Generating URL ...
Please wait ...
Data downloaded to I:\Dhaka_UTFVI_2005.tif
Generating URL ...
Please wait ...
Data downloaded to I:\Dhaka_UTCI_2005.tif
Processing 2015 Data...
Generating URL ...
Please wait ...
Data downloaded to I:\Dhaka_LST_2015.tif
Generating URL ...
Please wait ...
Data downloaded to I:\Dhaka_UTFVI_2015.tif
Generating URL ...
Please wait ...
Data downloaded to I:\Dhaka_UTCI_2015.tif
Processing 2025 Data...
Generating URL ...
Please wait ...
Data downloaded to I:\Dhaka_LST_2025.tif
Generating URL ...
Please wait ...
Data downloaded to I:\Dhaka_UTFVI_2025.tif
Generating URL ...
Please wait ...
Data downloaded to I:\Dhaka_UTCI_2025.tif
Generating Binned Trend Line Graphs...
Generating High-Resolution Maps...
Exported: Map_Standalone_LST_2005.jpg
Exported: Map_Standalone_UTFVI_2005.jpg
Exported: Map_Standalone_UTCI_2005.jpg
Exported: Map_Standalone_LST_2015.jpg
Export

In [13]:
# ==========================================
# GEE Python API: Master 4x3 Composite Grid 
# Memory-Safe Version (Using imshow)
# ==========================================

import pandas as pd
import matplotlib.pyplot as plt
import rioxarray

print("Generating Master 4x3 Publication Grid...")

# Set up the 4x3 grid (15x20 is large enough for high quality, but safe for memory)
fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(15, 20))
years = ['2005', '2015', '2025']
colors = ['blue', 'orange', 'darkred']

for i, year in enumerate(years):
    
    # --------------------------------------------------
    # ROW 1: LST Maps
    # --------------------------------------------------
    da_lst = rioxarray.open_rasterio(f'Dhaka_LST_{year}.tif').squeeze()
    da_lst = da_lst.where(da_lst > -9000)
    
    v_min, v_max = float(da_lst.quantile(0.02)), float(da_lst.quantile(0.98))
    # THE MEMORY FIX: Using .plot.imshow() instead of .plot()
    plot_lst = da_lst.plot.imshow(ax=axes[0, i], cmap='RdYlBu_r', vmin=v_min, vmax=v_max, add_colorbar=False)
    
    cbar = fig.colorbar(plot_lst, ax=axes[0, i], fraction=0.046, pad=0.04)
    cbar.set_label('Land Surface Temp (°C)', rotation=270, labelpad=15)
    axes[0, i].set_title(f'LST ({year})', fontweight='bold')

    # --------------------------------------------------
    # ROW 2: UTFVI Maps
    # --------------------------------------------------
    da_utfvi = rioxarray.open_rasterio(f'Dhaka_UTFVI_{year}.tif').squeeze()
    da_utfvi = da_utfvi.where(da_utfvi > -9000)
    
    v_min, v_max = float(da_utfvi.quantile(0.02)), float(da_utfvi.quantile(0.98))
    plot_utfvi = da_utfvi.plot.imshow(ax=axes[1, i], cmap='inferno', vmin=v_min, vmax=v_max, add_colorbar=False)
    
    cbar = fig.colorbar(plot_utfvi, ax=axes[1, i], fraction=0.046, pad=0.04)
    cbar.set_label('UTFVI', rotation=270, labelpad=15)
    axes[1, i].set_title(f'UTFVI ({year})', fontweight='bold')

    # --------------------------------------------------
    # ROW 3: UTCI Maps 
    # --------------------------------------------------
    da_utci = rioxarray.open_rasterio(f'Dhaka_UTCI_{year}.tif').squeeze()
    da_utci = da_utci.where(da_utci > -9000)
    
    v_min, v_max = float(da_utci.quantile(0.02)), float(da_utci.quantile(0.98))
    plot_utci = da_utci.plot.imshow(ax=axes[2, i], cmap='Spectral_r', vmin=v_min, vmax=v_max, add_colorbar=False)
    
    cbar = fig.colorbar(plot_utci, ax=axes[2, i], fraction=0.046, pad=0.04)
    cbar.set_label('UTCI (°C)', rotation=270, labelpad=15)
    axes[2, i].set_title(f'UTCI ({year})', fontweight='bold')

    # --------------------------------------------------
    # ROW 4: Correlation Scatter Plots
    # --------------------------------------------------
    df = pd.read_csv(f'Table_Dhaka_Data_{year}.csv')
    axes[3, i].scatter(df['UTFVI'], df['UTCI'], alpha=0.5, color=colors[i], edgecolor='black', s=20)
    
    axes[3, i].set_xlabel('UTFVI', fontsize=11)
    axes[3, i].set_ylabel('UTCI (°C)', fontsize=11)
    axes[3, i].set_title(f'Correlation ({year})', fontweight='bold')
    axes[3, i].grid(True, linestyle='--', alpha=0.6)
    
    axes[3, i].set_xlim([df['UTFVI'].min() - 0.02, df['UTFVI'].max() + 0.02])
    axes[3, i].set_ylim([df['UTCI'].min() - 0.5, df['UTCI'].max() + 0.5])

    # --------------------------------------------------
    # CARTOGRAPHIC ELEMENTS (Apply to Maps Only)
    # --------------------------------------------------
    for row in [0, 1, 2]:
        axes[row, i].set_xlabel('Longitude', fontsize=9)
        axes[row, i].set_ylabel('Latitude', fontsize=9)
        axes[row, i].set_facecolor('white') 
        
        # Simple, solid North Arrow
        axes[row, i].annotate('N', xy=(0.08, 0.95), xytext=(0.08, 0.88),
                              arrowprops=dict(facecolor='black', width=3, headwidth=10),
                              ha='center', va='center', fontsize=12, fontweight='bold',
                              xycoords=axes[row, i].transAxes, zorder=10, 
                              bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="none", alpha=0.8))

# Apply simple black borders (neatlines) to all graphs and maps
for row in range(4):
    for col in range(3):
        for spine in axes[row, col].spines.values():
            spine.set_edgecolor('black')
            spine.set_linewidth(1.2)

# Prevent overlap and save
plt.tight_layout(pad=2.0)
plt.savefig('Master_Grid_4x3_Dhaka.jpg', dpi=300, bbox_inches='tight')
plt.close()

print("Master Grid successfully generated and saved!")

Generating Master 4x3 Publication Grid...
Master Grid successfully generated and saved!
